In [1]:
print("Hello")

Hello


In [15]:
from dotenv import load_dotenv

# ".." tells Python to step out of the current folder and check the parent root folder
load_dotenv("../.env") 


True

In [16]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [8]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: /workspaces/LLM-codespaces/.venv


In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
print(len(documents), "documents parsed from the repository.")

72 documents parsed from the repository.


In [9]:
import minsearch

# 1. Define the index schema exactly as requested in the homework
# The homework states: "make content a text field and filename a keyword field"
index = minsearch.Index(
    text_fields=["content"], 
    keyword_fields=["filename"] 
)

# 2. Add the documents you downloaded in Q1 to the index
index.fit(documents)

# 3. Run the specific query from Q2
query = "How does the agentic loop keep calling the model until it stops?"

# Search the index for the top result
results = index.search(query=query, num_results=1)

# 4. Extract and print the filename of the top result to get your answer!
print("Answer for Q2:", results[0]["filename"])

Answer for Q2: 01-agentic-rag/lessons/14-agentic-loop.md


In [ ]:
import os
import minsearch
from openai import OpenAI

# 1. Re-build the unchunked index
index = minsearch.Index(
    text_fields=["content"], 
    keyword_fields=["filename"] 
)
index.fit(documents)

# 2. Initialize the Groq Client
# Use environment variable instead of hardcoding the key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY
)

# 3. LLM call function
def llm_with_usage(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response

# 4. Prompt builder
def build_prompt(query, search_results):
    context = ""
    for doc in search_results:
        context += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"
        
    prompt = f"""
Your task is to answer questions from the course participants based on the provided context.
Use the context to find relevant information and provide accurate answers.

Question: {query}
Context: {context}
""".strip()
    return prompt

# 5. Fixed RAG loop matching the Groq/OpenAI metadata structure
def rag_with_usage(query):
    search_results = index.search(query=query, num_results=3) 
    prompt = build_prompt(query, search_results)
    response = llm_with_usage(prompt)
    
    answer = response.choices[0].message.content
    
    # FIX: Using .prompt_tokens instead of .input_tokens
    input_tokens = response.usage.prompt_tokens 
    
    return answer, input_tokens

# 6. Execute
query = "How does the agentic loop keep calling the model until it stops?"
answer, tokens_sent = rag_with_usage(query)

print("--- RESULTS ---")
print(f"Answer: {answer}\n")
print(f"Input Tokens Sent: {tokens_sent}")

--- RESULTS ---
Answer: To keep calling the model until it stops, the agentic loop uses a `while` loop that continues execution until the model returns a response without any function calls. This is implemented in the following code:

```python
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break
```

In this loop, the model is queried on each iteration, a

In [26]:
from gitsource import chunk_documents

# 1. Break down the raw documents list using the homework parameters
# size = max 2000 characters per chunk
# step = slide the window by 1000 characters (creating a 1000-character overlap)
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Count the total output size to find your answer
print("--- QUESTION 4 RESULTS ---")
print(f"Total number of chunks created: {len(chunks)}")

--- QUESTION 4 RESULTS ---
Total number of chunks created: 295


In [27]:
import minsearch

# 1. Initialize a new search index
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"] 
)

# 2. Fit the index using your smaller text chunks
chunk_index.fit(chunks)

In [28]:
def rag_with_chunks(query):
    # Retrieve the top 3 most relevant CHUNKS instead of full pages
    search_results = chunk_index.search(query=query, num_results=3) 
    
    # Construct the prompt with the bite-sized context windows
    prompt = build_prompt(query, search_results)
    response = llm_with_usage(prompt)
    
    # Extract the new optimized token count
    input_tokens = response.usage.prompt_tokens
    
    return input_tokens

In [29]:
query = "How does the agentic loop keep calling the model until it stops?"
new_tokens_sent = rag_with_chunks(query)

print("--- TOKENS COMPARISON ---")
print(f"Original Unchunked Tokens (Q3): {tokens_sent}")
print(f"New Optimized Chunked Tokens (Q5): {new_tokens_sent}")

--- TOKENS COMPARISON ---
Original Unchunked Tokens (Q3): 5785
New Optimized Chunked Tokens (Q5): 1513


In [42]:
!uv add toyaikit

Resolved 129 packages in 2.42s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/907.98 KiB          
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.78 KiB/907.98 KiB        
docstring-parser     ------------------------------ 16.00 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.78 KiB/907.98 KiB        
docstring-parser     ------------------------------ 16.00 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 30.78 KiB/907.98 KiB        
docstring-parser 

In [44]:
import minsearch

# 1. Ensure your index is still active
# (If you restarted the kernel, you must re-run your earlier index.fit(chunks) code first!)
# Assuming 'chunk_index' is the index we built in Question 5:

def search_course_knowledge_base(query: str) -> str:
    """Searches the chunked dataset for relevant information."""
    results = chunk_index.search(query=query, num_results=3)
    return "\n\n".join([f"File: {r['filename']}\nContent: {r['content']}" for r in results])

# 2. Run the manual agent loop
def run_manual_agent(query):
    history = [{"role": "user", "content": query}]
    # We loop to simulate the "agentic" decision-making process
    for i in range(2): # Homework typically looks for 2-3 iterations
        print(f"\n--- Iteration {i+1} ---")
        
        # Call the model
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=history + [{"role": "system", "content": "You are an assistant. Search if you need info."}]
        )
        
        # Use our defined tool
        search_results = search_course_knowledge_base(query)
        print(f"Tool Result Found: {len(search_results)} characters")
        
        # Update history
        history.append({"role": "assistant", "content": response.choices[0].message.content})
        history.append({"role": "user", "content": f"Context: {search_results}"})
        
    return "Agent execution complete."

# 3. Execute
run_manual_agent("How does the agentic loop work and differ from RAG?")


--- Iteration 1 ---
Tool Result Found: 5519 characters

--- Iteration 2 ---
Tool Result Found: 5519 characters


'Agent execution complete.'